In [66]:
import torch
from torch.utils.data import Dataset
import os

class BeatmapChunkDataset(Dataset):
    def __init__(self, input_folder):
        self.audio_folder = os.path.join(input_folder, "audio")
        
        df = pd.read_csv(os.path.join(input_folder, "chunked.csv"))
        self.groups = list(df.groupby(["id", "chunk_id"]))

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        (beatmap_id, chunk_id), group = self.groups[idx]

        features = torch.tensor(
            group[["type_circle", "type_slider", "type_spinner", "repeat", "slider_velocity", 
                   "hit_start_rel", "hit_end_rel"]].values, dtype=torch.float32)
        beatmapset_id = beatmap_id.split("-")[0]
        chunk_audio_path = os.path.join(self.audio_folder, f"{beatmapset_id}_chunk{chunk_id}.pt")
        
        difficulty_rating = torch.tensor([group.iloc[0]["difficulty_rating"]], dtype=torch.float32)
        
        return {
            "beatmap_id": beatmap_id,
            "chunk_id": chunk_id,
            "features": features,
            "audio": torch.load(chunk_audio_path),
            "difficulty_rating": difficulty_rating 
        }


In [67]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    beatmap_ids = [item["beatmap_id"] for item in batch]
    chunk_ids = [item["chunk_id"] for item in batch]

    features_list = [item["features"] for item in batch]
    features_padded = pad_sequence(features_list, batch_first=True, padding_value=0.0)

    features_mask = torch.zeros(features_padded.shape[:2], dtype=torch.bool)
    for i, feat in enumerate(features_list):
        features_mask[i, :feat.shape[0]] = 1

    audio_embeddings = [item["audio"] for item in batch]
    difficulty_ratings = [item["difficulty_rating"] for item in batch]

    return {
        "beatmap_ids": beatmap_ids,
        "chunk_ids": torch.tensor(chunk_ids, dtype=torch.long),
        "features": features_padded,
        "features_mask": features_mask,
        "audio": audio_embeddings,
        "difficulty_rating" : difficulty_ratings
    }


In [68]:
import pandas as pd
from torch.utils.data import DataLoader

input_folder = "/home/saliherdemk/try_dataset/processed/"
dataset = BeatmapChunkDataset(input_folder)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn = collate_fn)

for batch in dataloader:
    print(batch)
    break

{'beatmap_ids': ['606419-4', '606419-1'], 'chunk_ids': tensor([10, 10]), 'features': tensor([[[0.0000, 1.0000, 0.0000, 0.3869, 0.1831, 0.0000, 0.0275],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1831, 0.0422, 0.0569],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0716, 0.0716],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0863, 0.0863],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1967, 0.1893, 0.2187],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1967, 0.2334, 0.2628],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2775, 0.2775],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1967, 0.3069, 0.3363],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3658, 0.3658],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3952, 0.3952],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1967, 0.4246, 0.4540],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.1967, 0.4687, 0.4981],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5128, 0.5128],
         [0.0000, 1.0000, 0.0000, 0.3869, 0.

# Model

In [69]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, num_layers=13, emb_dim=768):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=num_layers, out_channels=64, kernel_size=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=1, kernel_size=1)

    def forward(self, x, diff_rating):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = x.squeeze(1)
        x = torch.cat([x, diff_rating], dim = 1)
        return x


In [70]:
encoder = Encoder()
for batch in dataloader:
    audio_batch = torch.stack(batch["audio"])
    diff_batch = torch.stack(batch["difficulty_rating"])
    
    print(encoder(audio_batch, diff_batch).shape)
    break

torch.Size([2, 769])
